In [0]:
# Camada Gold - Modelagem dimensional com tabelas ponte

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "workspace"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"


def ler_silver(nome_tabela):
    return spark.read.table(f"{CATALOGO}.{SCHEMA_SILVER}.{nome_tabela}")


def ler_gold(nome_tabela):
    return spark.read.table(f"{CATALOGO}.{SCHEMA_GOLD}.{nome_tabela}")


def salvar_gold(df, nome_tabela):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(f"{CATALOGO}.{SCHEMA_GOLD}.{nome_tabela}")
    )


def gerar_surrogate_key(df, nome_sk, colunas_ordem):
    janela = Window.orderBy(*colunas_ordem)
    return df.withColumn(nome_sk, F.row_number().over(janela).cast("bigint"))

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.gold;

# Modelagem dimensional
## gold.dim_movies

In [0]:
df_info = ler_silver("tb_info_filmes")

df_dim_movies = (
    gerar_surrogate_key(df_info, "sk_movie_id", ["id_filme"])
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("id_filme").cast("string"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)

salvar_gold(df_dim_movies, "dim_movies")
display(ler_gold("dim_movies").limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null


## gold.fact_movies_performance

In [0]:
df_dim_movies = ler_gold("dim_movies")
df_financeiro = ler_silver("tb_financeiro_filmes")
df_metricas = ler_silver("tb_metricas_engajamento")

# 1 registro por filme lançado.
df_fact = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
    .join(df_financeiro, "id_filme", "left")
    .join(df_metricas, "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

# o grão não pode ter sido duplicado pelos joins
qtd_linhas = df_fact.count()
qtd_filmes = df_fact.select("sk_movie_id").distinct().count()
print(f"Linhas: {qtd_linhas} | Filmes distintos: {qtd_filmes}")
assert qtd_linhas == qtd_filmes, "Grão duplicado na tabela fato!"

salvar_gold(df_fact, "fact_movies_performance")

Linhas: 96261 | Filmes distintos: 96261


## gold.dim_genres

In [0]:
df_dim_genres = (
    gerar_surrogate_key(
        ler_silver("tb_generos").select("nome_genero").distinct(),
        "sk_genre_id",
        ["nome_genero"],
    )
    .select(F.col("sk_genre_id").cast("bigint"), F.col("nome_genero").cast("string"))
)

salvar_gold(df_dim_genres, "dim_genres")
display(ler_gold("dim_genres"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


## gold.dim_people

In [0]:
df_pessoas_empresas = ler_silver("tb_pessoas_empresas")

TIPOS_PESSOA = ["Ator", "Diretor", "Roteirista"]

df_dim_people = (
    gerar_surrogate_key(
        df_pessoas_empresas
        .filter(F.col("tipo_entidade").isin(TIPOS_PESSOA))
        .select(
            F.col("nome_entidade").alias("nome_pessoa"),
            F.col("tipo_entidade").alias("tipo_pessoa"),
        )
        .distinct(),
        "sk_person_id",
        ["nome_pessoa", "tipo_pessoa"],
    )
    .select(
        F.col("sk_person_id").cast("bigint"),
        F.col("nome_pessoa").cast("string"),
        F.col("tipo_pessoa").cast("string"),
    )
)

salvar_gold(df_dim_people, "dim_people")
display(ler_gold("dim_people").groupBy("tipo_pessoa").count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


tipo_pessoa,count
Ator,268881
Roteirista,84577
Diretor,63354


## gold.dim_companies

In [0]:
df_dim_companies = (
    gerar_surrogate_key(
        df_pessoas_empresas
        .filter(F.col("tipo_entidade") == "Produtora")
        .select(F.col("nome_entidade").alias("nome_produtora"))
        .distinct(),
        "sk_company_id",
        ["nome_produtora"],
    )
    .select(F.col("sk_company_id").cast("bigint"), F.col("nome_produtora").cast("string"))
)

salvar_gold(df_dim_companies, "dim_companies")
print(f"Produtoras: {ler_gold('dim_companies').count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Produtoras: 44383


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## gold.dim_reviews

In [0]:
df_resumo_reviews = (
    ler_silver("tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios"),
    )
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
)

df_dim_reviews = (
    gerar_surrogate_key(df_resumo_reviews, "sk_review_id", ["sk_movie_id"])
    .select(
        F.col("sk_review_id").cast("bigint"),
        F.col("sk_movie_id").cast("bigint"),
        F.col("qtd_avaliacoes_usuarios").cast("int"),
        F.col("nota_media_usuarios").cast("double"),
    )
)

salvar_gold(df_dim_reviews, "dim_reviews")
display(ler_gold("dim_reviews").limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,1,0.1
2,5,1,3.3
3,8,1,6.8
4,17,1,5.4
5,19,1,5.3


## gold.bridge_movie_genre

In [0]:
df_bridge_movie_genre = (
    ler_silver("tb_generos")
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .join(ler_gold("dim_genres"), "nome_genero", "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_genre_id").cast("bigint"))
    .distinct()
)

salvar_gold(df_bridge_movie_genre, "bridge_movie_genre")

## gold.bridge_movie_person

In [0]:
df_bridge_movie_person = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade").isin(TIPOS_PESSOA))
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .join(ler_gold("dim_people"), ["nome_pessoa", "tipo_pessoa"], "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_person_id").cast("bigint"))
    .distinct()
)

salvar_gold(df_bridge_movie_person, "bridge_movie_person")

In [0]:
df_bridge_movie_company = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .join(ler_gold("dim_companies"), "nome_produtora", "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_company_id").cast("bigint"))
    .distinct()
)

salvar_gold(df_bridge_movie_company, "bridge_movie_company")

In [0]:
tabelas_gold = [
    "fact_movies_performance", "dim_movies", "dim_genres", "dim_people",
    "dim_companies", "dim_reviews", "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company",
]

for nome_tabela in tabelas_gold:
    df_check = ler_gold(nome_tabela)
    print(f"\n{nome_tabela}: {df_check.count()} registros")
    df_check.printSchema()


fact_movies_performance: 96261 registros
root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)


dim_movies: 97611 registros
root
 |-- sk_movie_id: long (nullable = true)
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = t

# Desafio de Analytics

In [0]:
df_fact = ler_gold("fact_movies_performance")
df_dim_movies = ler_gold("dim_movies")

Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
display(
    df_fact.agg(F.sum("receita_brl").alias("receita_total_brl"))
    .withColumn("receita_total_brl_formatada", F.format_number("receita_total_brl", 2))
)

receita_total_brl,receita_total_brl_formatada
811127350229.87,"811,127,350,229.87"


Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.

In [0]:
display(
    df_fact
    .join(df_dim_movies, "sk_movie_id")
    .filter(F.col("popularidade").isNotNull())
    .orderBy(F.desc("popularidade"))
    .select("titulo", "popularidade")
    .limit(5)
)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


Quantos filmes cada gênero possui? Liste do maior para o menor volume.

In [0]:
display(
    ler_gold("bridge_movie_genre")
    .join(ler_gold("dim_genres"), "sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"))
)

nome_genero,qtd_filmes
Drama,32127
Documentary,18928
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
TV Movie,4066


Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking
(RANK()).

In [0]:
janela_receita = Window.orderBy(F.desc("receita_usd"))

display(
    df_fact
    .filter(F.col("receita_usd").isNotNull())
    .join(df_dim_movies, "sk_movie_id")
    .withColumn("posicao_ranking", F.rank().over(janela_receita))
    .filter(F.col("posicao_ranking") <= 10)
    .select("posicao_ranking", "titulo", "receita_usd", "receita_brl")
    .orderBy("posicao_ranking")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,13980120000.00
2,Avatar: The Way of Water,2320250281.00,11584777628.00
3,AVENGERS: INFINITY WAR,2052415039.00,10247503048.22
4,spider-man: no way home,1921847111.00,9595590440.51
5,The Lion King,1663075401.00,8303569169.65
6,Top Gun: Maverick,1488732821.00,7433094101.97
7,Barbie,1428545028.00,7132582470.30
8,The Super Mario Bros. Movie,1355725263.00,6769000665.63
9,Black Panther,1349926083.00,6740045939.81
10,Star Wars: The Last Jedi,1332698830.00,6654031988.31


Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?

In [0]:
df_filmes_lancados = df_dim_movies.filter(
    (F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date())
)

data_limite = df_filmes_lancados.agg(F.max("data_lancamento")).first()[0]
print(f"Data limite: {data_limite}")


Data limite: 2026-02-19


Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?

In [0]:
df_filmes_5_anos = df_filmes_lancados.filter(
    F.col("data_lancamento") >= F.add_months(F.lit(data_limite), -60)
)

display(
    ler_gold("bridge_movie_company")
    .join(ler_gold("dim_companies"), "sk_company_id")
    .join(df_filmes_5_anos.select("sk_movie_id"), "sk_movie_id")
    .join(df_fact.select("sk_movie_id", "lucro_usd", "lucro_brl"), "sk_movie_id")
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd"),
        F.sum("lucro_brl").alias("lucro_total_brl"),
    )
    .filter(F.col("lucro_total_usd").isNotNull())
    .orderBy(F.desc("lucro_total_usd"))
    .limit(5)
)

nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5772329679.00,28820664854.27
Marvel Studios,4953462823.00,24732144528.96
Columbia Pictures,3662050755.00,18284253214.65
Pascal Pictures,2701952454.00,13490578407.58
Illumination,2431353473.00,12139504755.34
